# L11 – Transfer learning

**Tid:** ca 2 h · **Del C: Nätverkstyper**

## Mål
1. Förstå **varför** en modell som tränats på miljontals bilder kan hjälpa till med *ditt* problem.
2. Använda en förtränad **ResNet18** med rätt förbehandling.
3. Skilja på **feature extraction** (frysa basen) och **fine-tuning** (träna vidare med låg learning rate).
4. Veta **när** du väljer vilket, utifrån datamängd och hur lik din data är förträningsdatan.

**Var i arbetsflödet?** Steg 3 och 5. För bildproblem med begränsad data är en förtränad modell ofta *både* den bästa baslinjen och den slutliga modellen.

> Kräver `dlkit` från L09. Första körningen laddar ned cirka 800 MB data (Oxford-IIIT Pets) och cirka 45 MB vikter. **Körtid:** cirka 6–8 minuter på GTX 1080.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from dlkit.seed import set_seed
from dlkit.train import fit, evaluate
from dlkit.plots import plot_history

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "../data"
loss_fn = nn.CrossEntropyLoss()

## 1. Problemet: lite data, svår uppgift

**Oxford-IIIT Pets** innehåller bilder av **37 hund- och kattraser**, med cirka 100 bilder per ras i träningsdelen. Det är *mycket* lite data för att lära sig skilja en Birma från en Ragdoll från grunden.

In [ ]:
trainval_raw = datasets.OxfordIIITPet(DATA_DIR, split="trainval", target_types="category", download=True)
classes = trainval_raw.classes
print(f"{len(trainval_raw)} bilder i trainval, {len(classes)} klasser")

fig, axes = plt.subplots(2, 6, figsize=(13, 4.5))
for ax, i in zip(axes.flat, np.random.default_rng(0).choice(len(trainval_raw), 12, replace=False)):
    img, label = trainval_raw[i]
    ax.imshow(img); ax.set_title(classes[label], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Idén bakom transfer learning

En CNN som tränats på **ImageNet** (1,2 miljoner bilder, 1 000 klasser) har lärt sig en hierarki av features (L10):
- tidiga lager: kanter, färger, texturer, vilket är *universellt* för alla foton,
- mellersta lager: former, ögon, päls, hjul,
- sena lager: delar av objekt som är specifika för ImageNet-klasserna.

**Transfer learning:** återanvänd `features`-delen och byt ut bara klassificeringshuvudet mot ett nytt för *dina* klasser. Modellen behöver då bara lära sig *hur* de färdiga features ska kombineras till 37 raser, och det går med lite data.

**Koppling till sklearn:** det liknar att använda en färdig `Pipeline`-transformator som någon annan har anpassat på mycket mer data, och sedan bara träna en `LogisticRegression` ovanpå.

### Ladda en förtränad ResNet18

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
preprocess = weights.transforms()
print("Förbehandlingen modellen tränades med:\n", preprocess)

model = models.resnet18(weights=weights)
print("\nSista lagret:", model.fc)
print(f"Parametrar: {sum(p.numel() for p in model.parameters()):,}")

**Viktigt: samma förbehandling som vid förträningen.** Modellen har lärt sig på bilder normaliserade med ImageNets medelvärde och standardavvikelse, i storleken 224×224. Ger du den annan skala blir alla features fel, med samma problem som när normaliseringen skiljer sig mellan train och test (L04). Därför används `weights.transforms()`, eller dess `mean` och `std`.

In [ ]:
MEAN, STD = preprocess.mean, preprocess.std

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),   # augmentation: slumpmässigt utsnitt
    transforms.RandomHorizontalFlip(),                     # en spegelvänd katt är fortfarande en katt
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Train/val från trainval (80/20) med två dataset-objekt, som i L10
perm = np.random.default_rng(42).permutation(len(trainval_raw))
n_train = int(0.8 * len(perm))
train_idx, val_idx = perm[:n_train], perm[n_train:]
train_ds = Subset(datasets.OxfordIIITPet(DATA_DIR, split="trainval", transform=train_tf), train_idx)
val_ds = Subset(datasets.OxfordIIITPet(DATA_DIR, split="trainval", transform=eval_tf), val_idx)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=4)
print(f"train: {len(train_ds)}  val: {len(val_ds)}")

## 3. Baslinje: en CNN från grunden

Först testar vi vad som händer *utan* förträning, med en CNN i stil med L10. `AdaptiveAvgPool2d(1)` tar medel över hela bilden, så att storleken in inte spelar roll.

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.MaxPool2d(2))

scratch = nn.Sequential(conv_block(3, 32), conv_block(32, 64), conv_block(64, 128), conv_block(128, 256),
                        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256, 37)).to(device)
set_seed(42)
h_scratch = fit(scratch, train_loader, val_loader, loss_fn,
                torch.optim.AdamW(scratch.parameters(), lr=1e-3, weight_decay=0.01), device, epochs=8)

Med cirka 80 bilder per klass når en CNN från grunden bara cirka **15–20 %** efter 8 epoker. Det är bättre än slumpen (2,7 %), men långt ifrån användbart. Datan räcker inte för att lära sig bra features från början.

## 4. Feature extraction: frys basen och träna bara huvudet

1. **Frys** alla förtränade vikter med `requires_grad = False`. Inga gradienter beräknas för dem, och de ändras inte.
2. **Byt** det sista lagret, `fc` (1000 ImageNet-klasser), mot ett nytt `Linear(512, 37)`. Nya lager har `requires_grad=True` som standard.
3. Ge **bara de tränbara parametrarna** till optimeraren.

In [ ]:
def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

set_seed(42)
model = models.resnet18(weights=weights)
for param in model.parameters():
    param.requires_grad = False                       # 1. frys allt
model.fc = nn.Linear(model.fc.in_features, 37)        # 2. nytt huvud, tränbart
model = model.to(device)
print(f"tränbara parametrar: {count_trainable(model):,} av {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)   # 3. bara huvudet
h_fe = fit(model, train_loader, val_loader, loss_fn, optimizer, device, epochs=5)

Med bara cirka 19 000 tränbara parametrar når vi cirka **85–88 %** på några epoker. Det är den sortens skillnad som gör transfer learning till standardvalet för bildproblem.

Det går också snabbt: backward behöver bara beräknas för det sista lagret.

## 5. Fine-tuning: träna vidare med låg learning rate

Nästa steg är att låta **hela nätet** anpassa sig lite till vår data. Raser skiljer sig i detaljer som ImageNet-modellen kanske inte prioriterat.

**Varför låg learning rate (1e-4)?** De förtränade vikterna är redan bra. Med hög learning rate skulle de stora stegen förstöra dem i början, vilket kallas *catastrophic forgetting*. Vi vill *justera*, inte lära om.

**Varför träna huvudet först?** Ett nytt, slumpmässigt huvud ger stora, meningslösa gradienter i början. Om basen är upplåst från början kan de gradienterna förstöra basen. Därför tränade vi huvudet först i avsnitt 4 och låser nu upp resten.

In [ ]:
for param in model.parameters():
    param.requires_grad = True                        # lås upp allt
print(f"tränbara parametrar: {count_trainable(model):,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
h_ft = fit(model, train_loader, val_loader, loss_fn, optimizer, device, epochs=4)

In [ ]:
print(f"{'metod':32s} {'bästa val acc':>13s}")
for name, h in [("CNN från grunden (8 epoker)", h_scratch),
                ("Feature extraction (5 epoker)", h_fe),
                ("+ fine-tuning (4 epoker)", h_ft)]:
    print(f"{name:32s} {max(h['val_acc']):>13.3f}")

## 6. Vad händer om … vi fine-tunar med hög learning rate?

**Gissa först:** vi tar en *ny* förtränad modell, låser upp allt direkt och tränar med `lr=1e-2` i stället för 1e-4.

In [ ]:
set_seed(42)
bad = models.resnet18(weights=weights)
bad.fc = nn.Linear(512, 37)
bad = bad.to(device)
h_bad = fit(bad, train_loader, val_loader, loss_fn,
            torch.optim.AdamW(bad.parameters(), lr=1e-2, weight_decay=0.01), device, epochs=3)

Resultatet blir katastrofalt: några få procent, alltså **sämre än CNN:en från grunden**. Lägg också märke till den enorma val loss efter första epoken. De stora stegen har förstört de förtränade features redan under de första batcherna. **Learning rate är ännu viktigare vid fine-tuning än vid vanlig träning.**

## 7. När väljer jag vad?

| Din situation | Strategi |
|---|---|
| **Lite data** (hundratals till några tusen bilder) och **liknande** ImageNet (foton av vardagliga objekt) | **Feature extraction**. Kanske lätt fine-tuning av de sista lagren. |
| **Mer data** (tiotusentals) eller **något annorlunda** data | **Fine-tuning** av hela nätet med låg learning rate, efter att huvudet tränats först |
| **Mycket annorlunda data** (medicinska bilder, spektrogram) | Fine-tuning hjälper oftast ändå, eftersom de tidiga lagren är generella. Jämför mot en modell från grunden om du har mycket data. |
| **Enorma mängder data** (miljoner) | Från grunden *kan* vara bättre, men förtränat är fortfarande en snabbare start. |

Samma idé gäller **text**: förtränade språkmodeller (BERT, GPT) finjusteras för specifika uppgifter. Vi nämner det igen i L13.

## 8. Test – en gång

In [ ]:
test_ds = datasets.OxfordIIITPet(DATA_DIR, split="test", target_types="category", transform=eval_tf, download=True)
test_loader = DataLoader(test_ds, batch_size=128, num_workers=4)
print(f"TEST accuracy (fine-tunad ResNet18): {evaluate(model, test_loader, loss_fn, device)[1]:.3f}")

## 9. Övningar

### Övning 1 – räkna tränbara parametrar
Hur många parametrar är tränbara i feature extraction-modellen (frusen bas och `Linear(512, 37)`)? Räkna för hand.

In [ ]:
my_count = None  # TODO

In [ ]:
m = models.resnet18(weights=None)
for p in m.parameters():
    p.requires_grad = False
m.fc = nn.Linear(512, 37)
assert my_count == count_trainable(m)
print("Rätt!", my_count)

### Övning 2 – lås upp bara det sista blocket
En mellanväg mellan feature extraction och full fine-tuning: frys allt **utom `layer4` och `fc`**. Träna i 4 epoker med `lr=1e-4`, med start från en modell där huvudet först tränats i 3 epoker med frusen bas. Hur står det sig mot full fine-tuning?

*Ledtråd:* `model.layer4.parameters()`.

In [ ]:
# TODO
h_layer4 = None

In [ ]:
assert max(h_layer4["val_acc"]) > 0.85
print("Rätt!", max(h_layer4["val_acc"]))

### Övning 3 – en annan förtränad modell
Testa `models.mobilenet_v3_small` (`MobileNet_V3_Small_Weights.DEFAULT`) med feature extraction. Huvudet heter `classifier`, och det sista lagret är `classifier[3]`. Jämför accuracy, antal parametrar och tid per epok med ResNet18. När skulle du välja MobileNet?

In [ ]:
# TODO

*Din jämförelse:*

## När använder jag detta?
- **Transfer learning:** som *förstahandsval* för bilder (och text) när du inte har hundratusentals exempel.
- **Feature extraction:** när du har lite data, eller för en snabb och stark baslinje.
- **Fine-tuning:** när du har mer data eller vill pressa ut de sista procenten. Träna huvudet först och använd låg learning rate (cirka 1e-4).
- **Förbehandling:** alltid samma som modellen förtränades med (`weights.transforms()`).
- **Modellval:** ResNet är ett robust standardval, MobileNet passar när modellen ska vara liten och snabb (mobil, CPU), och större modeller passar när accuracy är allt.

## Sammanfattning
- Förtränade features är generella. Med ett nytt huvud räcker lite data.
- Feature extraction innebär att frysa basen och träna huvudet. Fine-tuning innebär att låsa upp och träna allt med låg learning rate.
- Från grunden cirka 15–20 %, med transfer learning cirka 90 %, på samma data.

**Nästa lektion:** sekvenser. RNN och LSTM för tidsserier.